In [ ]:
# here the goal is to build he main analysis-ready datasets by joining
# daily beach activity, rainfall, river-flow, H₂S, and spatial variables
from pathlib import Path
import pandas as pd
import numpy as np

try:
    import geopandas as gpd
except ImportError:
    gpd = None

DATA_RAW = Path("/content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles")
DATA_PROCESSED = DATA_RAW / "processed_data"
FIGURES = DATA_RAW / "figures"
TABLES = DATA_RAW / "tables"
NOTEBOOKS = DATA_RAW / "notebooks"

for folder in [DATA_PROCESSED, FIGURES, TABLES, NOTEBOOKS]:
    folder.mkdir(parents=True, exist_ok=True)

paths = {
    "rainfall": DATA_PROCESSED / "prism_daily_rainfall_south_bay.csv",
    "daily_beach": DATA_PROCESSED / "daily_beach_closure_activity.csv",
    "beach_long": DATA_PROCESSED / "daily_beach_activity_long.csv",
    "river_flow": DATA_PROCESSED / "tijuana_river_daily_flow_2023_2026.csv",
    "h2s_events": DATA_PROCESSED / "h2s_events_clean_2024_2025.csv",
    "h2s_daily": DATA_PROCESSED / "h2s_daily_station_summary_2024_2025.csv",
    "h2s_rtma": DATA_PROCESSED / "h2s_rtma_matched_events_clean.csv",
    "beach_stations": DATA_PROCESSED / "beach_monitoring_stations_clean.csv",
    "census_attributes": DATA_PROCESSED / "south_bay_census_tracts_proximity_attributes.csv",
    "census_geojson": DATA_PROCESSED / "south_bay_census_tracts_proximity.geojson",
}

for name, path in paths.items():
    print(f"{name}: {path.exists()} | {path}")

rainfall = pd.read_csv(paths["rainfall"])
daily_beach = pd.read_csv(paths["daily_beach"])
beach_long = pd.read_csv(paths["beach_long"])
river_flow = pd.read_csv(paths["river_flow"])
h2s_events = pd.read_csv(paths["h2s_events"])
h2s_daily = pd.read_csv(paths["h2s_daily"])
h2s_rtma = pd.read_csv(paths["h2s_rtma"])
beach_stations = pd.read_csv(paths["beach_stations"])
census_attributes = pd.read_csv(paths["census_attributes"], dtype={"geoid": str})

if gpd is not None:
    census_geojson = gpd.read_file(paths["census_geojson"])
else:
    census_geojson = None

datasets = {
    "rainfall": rainfall,
    "daily_beach": daily_beach,
    "beach_long": beach_long,
    "river_flow": river_flow,
    "h2s_events": h2s_events,
    "h2s_daily": h2s_daily,
    "h2s_rtma": h2s_rtma,
    "beach_stations": beach_stations,
    "census_attributes": census_attributes,
}

for name, df in datasets.items():
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("\nBasic input file checks")
print("=" * 70)

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 70)
    print(f"Rows: {len(df)}")
    print(f"Columns: {df.shape[1]}")

    if "date" in df.columns:
        print(f"Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"Unique dates: {df['date'].nunique()}")
        print(f"Missing dates: {df['date'].isna().sum()}")

    if "station_id" in df.columns:
        print(f"Unique station_id values: {df['station_id'].nunique()}")
        print(f"Missing station_id values: {df['station_id'].isna().sum()}")

    if "geoid" in df.columns:
        print("GEOID length counts:")
        print(df["geoid"].dropna().astype(str).str.len().value_counts().sort_index())

if census_geojson is not None:
    print("\ncensus_geojson")
    print("-" * 70)
    print(f"Rows: {len(census_geojson)}")
    print(f"Columns: {census_geojson.shape[1]}")
    print(f"CRS: {census_geojson.crs}")
else:
    print("\ncensus_geojson was not loaded because geopandas is unavailable.")

rainfall: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/prism_daily_rainfall_south_bay.csv
daily_beach: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_closure_activity.csv
beach_long: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/daily_beach_activity_long.csv
river_flow: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/tijuana_river_daily_flow_2023_2026.csv
h2s_events: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_events_clean_2024_2025.csv
h2s_daily: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_daily_station_summary_2024_2025.csv
h2s_rtma: True | /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_rtma_matched_events_clean.csv
beach_stations: True | /content/gdrive_clean_mount/MyDrive/STAT596/Pr

In [ ]:
# Confirms rainfall, beach activity, and river flow can be safely joined by date

main_daily_inputs = {
    "daily_beach": daily_beach,
    "rainfall": rainfall,
    "river_flow": river_flow,
}

print("Duplicate date checks")
print("=" * 70)

for name, df in main_daily_inputs.items():
    duplicate_dates = df["date"].duplicated().sum()
    print(f"{name}: duplicate dates = {duplicate_dates}")

print("\nDate alignment checks")
print("=" * 70)

daily_dates = set(daily_beach["date"])
rain_dates = set(rainfall["date"])
flow_dates = set(river_flow["date"])

print("daily_beach dates == rainfall dates:", daily_dates == rain_dates)
print("daily_beach dates == river_flow dates:", daily_dates == flow_dates)
print("rainfall dates == river_flow dates:", rain_dates == flow_dates)

print("\nDates missing from rainfall compared to daily_beach:", len(daily_dates - rain_dates))
print("Dates missing from river_flow compared to daily_beach:", len(daily_dates - flow_dates))
print("Extra rainfall dates:", len(rain_dates - daily_dates))
print("Extra river_flow dates:", len(flow_dates - daily_dates))

print("\nOverlapping column checks besides date")
print("=" * 70)

daily_cols = set(daily_beach.columns)
rain_cols = set(rainfall.columns)
flow_cols = set(river_flow.columns)

overlap_daily_rain = sorted((daily_cols & rain_cols) - {"date"})
overlap_daily_flow = sorted((daily_cols & flow_cols) - {"date"})
overlap_rain_flow = sorted((rain_cols & flow_cols) - {"date"})

print("daily_beach + rainfall overlaps:")
print(overlap_daily_rain)

print("\ndaily_beach + river_flow overlaps:")
print(overlap_daily_flow)

print("\nrainfall + river_flow overlaps:")
print(overlap_rain_flow)

print("\nMain input shapes")
print("=" * 70)
print("daily_beach:", daily_beach.shape)
print("rainfall:", rainfall.shape)
print("river_flow:", river_flow.shape)

Duplicate date checks
daily_beach: duplicate dates = 0
rainfall: duplicate dates = 0
river_flow: duplicate dates = 0

Date alignment checks
daily_beach dates == rainfall dates: True
daily_beach dates == river_flow dates: True
rainfall dates == river_flow dates: True

Dates missing from rainfall compared to daily_beach: 0
Dates missing from river_flow compared to daily_beach: 0
Extra rainfall dates: 0
Extra river_flow dates: 0

Overlapping column checks besides date
daily_beach + rainfall overlaps:
[]

daily_beach + river_flow overlaps:
[]

rainfall + river_flow overlaps:
[]

Main input shapes
daily_beach: (1159, 41)
rainfall: (1159, 6)
river_flow: (1159, 32)


In [ ]:
#Creating  the main daily file for beach activity, rainfall,
#  and Tijuana River flow analysis.

daily_environmental = (
    daily_beach
    .merge(rainfall, on="date", how="left")
    .merge(river_flow, on="date", how="left")
)

daily_environmental_path = DATA_PROCESSED / "daily_environmental_analysis_ready.csv"

daily_environmental.to_csv(daily_environmental_path, index=False)

daily_environmental_check = pd.read_csv(daily_environmental_path)
daily_environmental_check["date"] = pd.to_datetime(daily_environmental_check["date"], errors="coerce")

print("Saved main daily analysis-ready file")
print("=" * 70)
print("Path:", daily_environmental_path)
print("File exists:", daily_environmental_path.exists())

print("\nReloaded QA")
print("=" * 70)
print("Rows:", len(daily_environmental_check))
print("Columns:", daily_environmental_check.shape[1])
print("Date range:", daily_environmental_check["date"].min(), "to", daily_environmental_check["date"].max())
print("Unique dates:", daily_environmental_check["date"].nunique())
print("Duplicate dates:", daily_environmental_check["date"].duplicated().sum())

key_fields = [
    "date",
    "rain_mm",
    "rain_lag1_mm",
    "rain_3day_mm",
    "rain_7day_mm",
    "rainy_day",
    "discharge_m3s",
    "discharge_cfs",
    "discharge_mgd",
    "high_flow_day",
    "flow_category",
    "flow_lag1_m3s",
    "flow_3day_mean_m3s",
    "flow_7day_mean_m3s",
    "high_flow_1day_window",
    "high_flow_3day_window",
    "high_flow_7day_window",
]

existing_key_fields = [col for col in key_fields if col in daily_environmental_check.columns]
missing_key_fields = [col for col in key_fields if col not in daily_environmental_check.columns]

print("\nMissing expected key fields:")
print(missing_key_fields)

print("\nMissing values in existing key fields:")
print(daily_environmental_check[existing_key_fields].isna().sum())

print("\nFirst 5 rows:")
display(daily_environmental_check.head())

Saved main daily analysis-ready file
Path: /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/daily_environmental_analysis_ready.csv
File exists: True

Reloaded QA
Rows: 1159
Columns: 77
Date range: 2023-01-01 00:00:00 to 2026-03-04 00:00:00
Unique dates: 1159
Duplicate dates: 0

Missing expected key fields:
[]

Missing values in existing key fields:
date                     0
rain_mm                  0
rain_lag1_mm             1
rain_3day_mm             0
rain_7day_mm             0
rainy_day                0
discharge_m3s            0
discharge_cfs            0
discharge_mgd            0
high_flow_day            0
flow_category            0
flow_lag1_m3s            1
flow_3day_mean_m3s       0
flow_7day_mean_m3s       0
high_flow_1day_window    0
high_flow_3day_window    0
high_flow_7day_window    0
dtype: int64

First 5 rows:


,date,active_advisory_records,active_closure_records,active_posting_records,active_rain_advisory_records,active_bacteria_related_records,active_tijuana_river_related_records,active_sewage_infrastructure_records,active_sewage_related_records,active_enterococcus_records,...,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
0,2023-01-01,6,3,2,1,5,3,0,3,5,...,78.300000,78.3000,1,1,1,-177,0,0,0,Before emergency reference window
1,2023-01-02,7,4,2,1,6,4,0,4,6,...,43.880000,43.8800,1,1,1,-176,0,0,0,Before emergency reference window
2,2023-01-03,6,4,1,1,5,4,0,4,5,...,30.950000,30.9500,1,1,1,-175,0,0,0,Before emergency reference window
3,2023-01-04,6,4,1,1,5,4,0,4,5,...,6.023333,24.0925,1,1,1,-174,0,0,0,Before emergency reference window
4,2023-01-05,6,4,1,1,5,4,0,4,5,...,4.123333,20.0260,1,1,1,-173,0,0,0,Before emergency reference window


In [ ]:
# Confirms H2S daily station records can be linked to
# rainfall, beach activity, and river flow by date

daily_environmental = daily_environmental_check.copy()

print("H2S daily station-date checks")
print("=" * 70)

h2s_station_cols = [col for col in h2s_daily.columns if "station" in col.lower() or "location" in col.lower()]
print("Possible station/location columns:")
print(h2s_station_cols)

station_col = None

for col in ["location_clean", "station_name", "station_id", "location"]:
    if col in h2s_daily.columns:
        station_col = col
        break

print("\nSelected station column:", station_col)

if station_col is not None:
    duplicate_station_dates = h2s_daily.duplicated(subset=["date", station_col]).sum()
    print("Duplicate station/date rows:", duplicate_station_dates)
    print("Unique stations:", h2s_daily[station_col].nunique())
    print("\nStation counts:")
    print(h2s_daily[station_col].value_counts())
else:
    print("No clear station column found. Inspect columns before joining.")

print("\nDate coverage checks")
print("=" * 70)

daily_env_dates = set(daily_environmental["date"])
h2s_daily_dates = set(h2s_daily["date"])

print("H2S daily dates inside daily environmental dates:", h2s_daily_dates.issubset(daily_env_dates))
print("H2S daily unique dates:", h2s_daily["date"].nunique())
print("Dates missing from daily environmental:", len(h2s_daily_dates - daily_env_dates))

print("\nOverlapping column checks besides date")
print("=" * 70)

overlap_h2s_daily_env = sorted((set(h2s_daily.columns) & set(daily_environmental.columns)) - {"date"})

print("h2s_daily + daily_environmental overlaps:")
print(overlap_h2s_daily_env)

print("\nInput shapes")
print("=" * 70)
print("h2s_daily:", h2s_daily.shape)
print("daily_environmental:", daily_environmental.shape)

print("\nH2S daily columns")
print("=" * 70)
print(list(h2s_daily.columns))

H2S daily station-date checks
Possible station/location columns:
['location_clean', 'station_name_clean']

Selected station column: location_clean
Duplicate station/date rows: 0
Unique stations: 3

Station counts:
location_clean
SAN YSIDRO      23
NESTOR - BES    22
IB CIVIC CTR    14
Name: count, dtype: int64

Date coverage checks
H2S daily dates inside daily environmental dates: True
H2S daily unique dates: 26
Dates missing from daily environmental: 0

Overlapping column checks besides date
h2s_daily + daily_environmental overlaps:
[]

Input shapes
h2s_daily: (59, 20)
daily_environmental: (1159, 77)

H2S daily columns
['date', 'location_clean', 'station_name_clean', 'first_datetime_local', 'last_datetime_local', 'total_records', 'valid_h2s_records', 'missing_or_invalid_records', 'mean_h2s_ppb', 'median_h2s_ppb', 'max_h2s_ppb', 'min_h2s_ppb', 'elevated_h2s_events', 'orange_h2s_events', 'exceedance_200ppb_events', 'nighttime_records', 'elevated_h2s_rate', 'orange_h2s_rate', 'any_elevat

In [ ]:
# Links daily H2S station summaries with rainfall,
# beach activity, and Tijuana River flow


h2s_daily_environmental = h2s_daily.merge(
    daily_environmental,
    on="date",
    how="left"
)

h2s_daily_environmental_path = DATA_PROCESSED / "h2s_daily_environmental_analysis_ready.csv"

h2s_daily_environmental.to_csv(h2s_daily_environmental_path, index=False)

h2s_daily_environmental_check = pd.read_csv(h2s_daily_environmental_path)
h2s_daily_environmental_check["date"] = pd.to_datetime(
    h2s_daily_environmental_check["date"],
    errors="coerce"
)

print("Saved H2S daily environmental analysis-ready file")
print("=" * 70)
print("Path:", h2s_daily_environmental_path)
print("File exists:", h2s_daily_environmental_path.exists())

print("\nReloaded QA")
print("=" * 70)
print("Rows:", len(h2s_daily_environmental_check))
print("Columns:", h2s_daily_environmental_check.shape[1])
print("Date range:", h2s_daily_environmental_check["date"].min(), "to", h2s_daily_environmental_check["date"].max())
print("Unique dates:", h2s_daily_environmental_check["date"].nunique())

print("\nStation/date QA")
print("=" * 70)
print("Selected station column:", station_col)

if station_col is not None:
    print("Unique stations:", h2s_daily_environmental_check[station_col].nunique())
    print("Duplicate station/date rows:", h2s_daily_environmental_check.duplicated(subset=["date", station_col]).sum())
    print("\nStation counts:")
    print(h2s_daily_environmental_check[station_col].value_counts())

key_fields = [
    "date",
    station_col,
    "mean_h2s_ppb",
    "max_h2s_ppb",
    "elevated_h2s_events",
    "orange_h2s_events",
    "any_elevated_h2s_day",
    "any_orange_h2s_day",
    "rain_mm",
    "rain_lag1_mm",
    "rain_3day_mm",
    "rain_7day_mm",
    "rainy_day",
    "discharge_m3s",
    "high_flow_day",
    "flow_category",
    "high_flow_1day_window",
    "high_flow_3day_window",
    "high_flow_7day_window",
]

key_fields = [col for col in key_fields if col is not None]
existing_key_fields = [col for col in key_fields if col in h2s_daily_environmental_check.columns]
missing_key_fields = [col for col in key_fields if col not in h2s_daily_environmental_check.columns]

print("\nMissing expected key fields:")
print(missing_key_fields)

print("\nMissing values in existing key fields:")
print(h2s_daily_environmental_check[existing_key_fields].isna().sum())

print("\nFirst 5 rows:")
display(h2s_daily_environmental_check.head())

Saved H2S daily environmental analysis-ready file
Path: /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_daily_environmental_analysis_ready.csv
File exists: True

Reloaded QA
Rows: 59
Columns: 96
Date range: 2024-09-12 00:00:00 to 2025-08-26 00:00:00
Unique dates: 26

Station/date QA
Selected station column: location_clean
Unique stations: 3
Duplicate station/date rows: 0

Station counts:
location_clean
SAN YSIDRO      23
NESTOR - BES    22
IB CIVIC CTR    14
Name: count, dtype: int64

Missing expected key fields:
[]

Missing values in existing key fields:
date                      0
location_clean            0
mean_h2s_ppb             21
max_h2s_ppb              21
elevated_h2s_events       0
orange_h2s_events         0
any_elevated_h2s_day      0
any_orange_h2s_day        0
rain_mm                   0
rain_lag1_mm              0
rain_3day_mm              0
rain_7day_mm              0
rainy_day                 0
discharge_m3s             0
high_flow_

,date,location_clean,station_name_clean,first_datetime_local,last_datetime_local,total_records,valid_h2s_records,missing_or_invalid_records,mean_h2s_ppb,median_h2s_ppb,...,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period,analysis_period
0,2024-09-12,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-12 11:00:00,2024-09-12 11:00:00,1,0,1,NaN,NaN,...,0.600000,1.567143,0,0,0,443,1,0,0,After emergency reference window
1,2024-09-25,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-25 14:00:00,2024-09-25 23:00:00,10,10,0,2.070000,0.15,...,0.266667,0.271429,0,0,0,456,1,0,1,After emergency reference window
2,2024-09-26,SAN YSIDRO,San Ysidro - Fire Station #29,2024-09-26 00:00:00,2024-09-26 14:00:00,15,15,0,0.126667,0.10,...,0.233333,0.255714,0,0,0,457,1,0,1,After emergency reference window
3,2024-10-12,NESTOR - BES,Nestor - Berry Elementary School,2024-10-12 06:00:00,2024-10-12 14:00:00,9,0,9,NaN,NaN,...,0.276667,0.280000,0,0,0,473,1,0,1,After emergency reference window
4,2024-10-12,SAN YSIDRO,San Ysidro - Fire Station #29,2024-10-12 06:00:00,2024-10-12 14:00:00,9,9,0,1.766667,1.00,...,0.276667,0.280000,0,0,0,473,1,0,1,After emergency reference window


In [ ]:
# H2S + RTMA event subset for analysis-ready integration

print("H2S RTMA event-level checks")
print("=" * 70)

h2s_rtma_station_cols = [col for col in h2s_rtma.columns if "station" in col.lower() or "location" in col.lower()]
print("Possible station/location columns:")
print(h2s_rtma_station_cols)

rtma_station_col = None

for col in ["location_clean", "station_name", "station_name_clean", "station_id", "location"]:
    if col in h2s_rtma.columns:
        rtma_station_col = col
        break

print("\nSelected station column:", rtma_station_col)

datetime_cols = [col for col in h2s_rtma.columns if "datetime" in col.lower() or "time" in col.lower()]
print("\nPossible datetime/time columns:")
print(datetime_cols)

print("\nDate coverage checks")
print("=" * 70)

h2s_rtma_dates = set(h2s_rtma["date"])
daily_env_dates = set(daily_environmental["date"])

print("H2S RTMA dates inside daily environmental dates:", h2s_rtma_dates.issubset(daily_env_dates))
print("H2S RTMA unique dates:", h2s_rtma["date"].nunique())
print("Dates missing from daily environmental:", len(h2s_rtma_dates - daily_env_dates))

print("\nStation/date counts")
print("=" * 70)

if rtma_station_col is not None:
    print("Unique stations:", h2s_rtma[rtma_station_col].nunique())
    print("\nStation counts:")
    print(h2s_rtma[rtma_station_col].value_counts())
    print("\nRows per station/date preview:")
    display(
        h2s_rtma
        .groupby(["date", rtma_station_col])
        .size()
        .reset_index(name="event_rows")
        .sort_values(["date", rtma_station_col])
        .head(20)
    )

print("\nOverlapping column checks besides date")
print("=" * 70)

overlap_h2s_rtma_env = sorted((set(h2s_rtma.columns) & set(daily_environmental.columns)) - {"date"})

print("h2s_rtma + daily_environmental overlaps:")
print(overlap_h2s_rtma_env)

print("\nInput shapes")
print("=" * 70)
print("h2s_rtma:", h2s_rtma.shape)
print("daily_environmental:", daily_environmental.shape)

print("\nH2S RTMA columns")
print("=" * 70)
print(list(h2s_rtma.columns))


H2S RTMA event-level checks
Possible station/location columns:
['location_clean']

Selected station column: location_clean

Possible datetime/time columns:
['datetime_local', 'datetime_utc', 'nighttime_flag', 'time', 'datetime_utc_original']

Date coverage checks
H2S RTMA dates inside daily environmental dates: True
H2S RTMA unique dates: 11
Dates missing from daily environmental: 0

Station/date counts
Unique stations: 3

Station counts:
location_clean
NESTOR - BES    31
SAN YSIDRO      20
IB CIVIC CTR     9
Name: count, dtype: int64

Rows per station/date preview:


,date,location_clean,event_rows
0,2024-09-25,SAN YSIDRO,1
1,2024-12-07,NESTOR - BES,8
2,2024-12-07,SAN YSIDRO,5
3,2024-12-16,NESTOR - BES,3
4,2024-12-25,NESTOR - BES,3
5,2024-12-26,NESTOR - BES,1
6,2025-01-16,IB CIVIC CTR,2
7,2025-01-31,NESTOR - BES,1
8,2025-05-18,NESTOR - BES,1
9,2025-05-20,IB CIVIC CTR,3



Overlapping column checks besides date
h2s_rtma + daily_environmental overlaps:
['analysis_period', 'month', 'year']

Input shapes
h2s_rtma: (60, 61)
daily_environmental: (1159, 77)

H2S RTMA columns
['datetime_local', 'datetime_utc', 'date', 'year', 'month', 'hour', 'location_clean', 'target_lat', 'target_lon', 'h2s_ppb', 'h2s_level_clean', 'valid_h2s_record', 'elevated_h2s_event', 'h2s_ge_10_event', 'orange_h2s_event', 'temperature_2m_f', 'dewpoint_2m_f', 'wind_speed_10m_mph', 'wind_direction_10m', 'wind_gust_10m_mph', 'visibility_surface_miles', 'temperature_2m', 'dewpoint_2m', 'wind_u_10m', 'wind_v_10m', 'wind_speed_10m', 'wind_gust_10m', 'visibility_surface', 'nighttime_flag', 'analysis_period', 'rtma_match_subset_definition', 'temperature_2m_units', 'temperature_2m_grid_lat', 'temperature_2m_grid_lon', 'dewpoint_2m_units', 'dewpoint_2m_grid_lat', 'dewpoint_2m_grid_lon', 'wind_u_10m_units', 'wind_u_10m_grid_lat', 'wind_u_10m_grid_lon', 'wind_v_10m_units', 'wind_v_10m_grid_lat', '

In [ ]:
#  H2S >= 10 ppb RTMA subset with rainfall, beach activity, and Tijuana River flow.

daily_environmental_for_rtma = daily_environmental.drop(
    columns=["year", "month", "analysis_period"],
    errors="ignore"
)

h2s_rtma_environmental = h2s_rtma.merge(
    daily_environmental_for_rtma,
    on="date",
    how="left"
)

h2s_rtma_environmental_path = DATA_PROCESSED / "h2s_rtma_environmental_analysis_ready.csv"

h2s_rtma_environmental.to_csv(h2s_rtma_environmental_path, index=False)

h2s_rtma_environmental_check = pd.read_csv(h2s_rtma_environmental_path)
h2s_rtma_environmental_check["date"] = pd.to_datetime(
    h2s_rtma_environmental_check["date"],
    errors="coerce"
)

print("Saved H2S RTMA environmental analysis-ready file")
print("=" * 70)
print("Path:", h2s_rtma_environmental_path)
print("File exists:", h2s_rtma_environmental_path.exists())

print("\nReloaded QA")
print("=" * 70)
print("Rows:", len(h2s_rtma_environmental_check))
print("Columns:", h2s_rtma_environmental_check.shape[1])
print("Date range:", h2s_rtma_environmental_check["date"].min(), "to", h2s_rtma_environmental_check["date"].max())
print("Unique dates:", h2s_rtma_environmental_check["date"].nunique())

print("\nStation/event QA")
print("=" * 70)
print("Selected station column:", rtma_station_col)

if rtma_station_col is not None:
    print("Unique stations:", h2s_rtma_environmental_check[rtma_station_col].nunique())
    print("\nStation counts:")
    print(h2s_rtma_environmental_check[rtma_station_col].value_counts())

print("\nDuplicate column suffix check")
print("=" * 70)
suffix_cols = [
    col for col in h2s_rtma_environmental_check.columns
    if col.endswith("_x") or col.endswith("_y")
]
print("Columns ending in _x or _y:")
print(suffix_cols)

key_fields = [
    "date",
    rtma_station_col,
    "datetime_local",
    "datetime_utc",
    "h2s_ppb",
    "h2s_level_clean",
    "h2s_ge_10_event",
    "orange_h2s_event",
    "temperature_2m_f",
    "dewpoint_2m_f",
    "wind_speed_10m_mph",
    "wind_direction_10m",
    "wind_gust_10m_mph",
    "visibility_surface_miles",
    "rain_mm",
    "rain_lag1_mm",
    "rain_3day_mm",
    "rain_7day_mm",
    "rainy_day",
    "discharge_m3s",
    "high_flow_day",
    "flow_category",
    "high_flow_1day_window",
    "high_flow_3day_window",
    "high_flow_7day_window",
]

key_fields = [col for col in key_fields if col is not None]
existing_key_fields = [col for col in key_fields if col in h2s_rtma_environmental_check.columns]
missing_key_fields = [col for col in key_fields if col not in h2s_rtma_environmental_check.columns]

print("\nMissing expected key fields:")
print(missing_key_fields)

print("\nMissing values in existing key fields:")
print(h2s_rtma_environmental_check[existing_key_fields].isna().sum())

print("\nReminder:")
print("This file is the stricter H2S >= 10 ppb RTMA-matched subset, not the full H2S dataset.")

print("\nFirst 5 rows:")
display(h2s_rtma_environmental_check.head())

Saved H2S RTMA environmental analysis-ready file
Path: /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/h2s_rtma_environmental_analysis_ready.csv
File exists: True

Reloaded QA
Rows: 60
Columns: 134
Date range: 2024-09-25 00:00:00 to 2025-07-08 00:00:00
Unique dates: 11

Station/event QA
Selected station column: location_clean
Unique stations: 3

Station counts:
location_clean
NESTOR - BES    31
SAN YSIDRO      20
IB CIVIC CTR     9
Name: count, dtype: int64

Duplicate column suffix check
Columns ending in _x or _y:
[]

Missing expected key fields:
[]

Missing values in existing key fields:
date                        0
location_clean              0
datetime_local              0
datetime_utc                0
h2s_ppb                     0
h2s_level_clean             0
h2s_ge_10_event             0
orange_h2s_event            0
temperature_2m_f            0
dewpoint_2m_f               0
wind_speed_10m_mph          0
wind_direction_10m          0
wind_gust_1

,datetime_local,datetime_utc,date,year,month,hour,location_clean,target_lat,target_lon,h2s_ppb,...,flow_lag1_m3s,flow_3day_mean_m3s,flow_7day_mean_m3s,high_flow_1day_window,high_flow_3day_window,high_flow_7day_window,days_from_emergency_declaration,post_emergency,emergency_reference_window,h2s_overlap_period
0,2024-09-25 19:00:00,2024-09-26 02:00:00,2024-09-25,2024,2024-09,19,SAN YSIDRO,32.552,-117.0431,10.0,...,0.23,0.266667,0.271429,0,0,0,456,1,0,1
1,2024-12-07 00:00:00,2024-12-07 08:00:00,2024-12-07,2024,2024-12,0,NESTOR - BES,32.579,-117.0910,17.8,...,0.55,0.573333,0.604286,0,0,0,529,1,0,1
2,2024-12-07 00:00:00,2024-12-07 08:00:00,2024-12-07,2024,2024-12,0,SAN YSIDRO,32.552,-117.0431,10.8,...,0.55,0.573333,0.604286,0,0,0,529,1,0,1
3,2024-12-07 01:00:00,2024-12-07 09:00:00,2024-12-07,2024,2024-12,1,NESTOR - BES,32.579,-117.0910,20.1,...,0.55,0.573333,0.604286,0,0,0,529,1,0,1
4,2024-12-07 01:00:00,2024-12-07 09:00:00,2024-12-07,2024,2024-12,1,SAN YSIDRO,32.552,-117.0431,15.0,...,0.55,0.573333,0.604286,0,0,0,529,1,0,1


In [ ]:
# again Confirming daily beach activity records can be
# linked to beach monitoring station metadata

print("Beach activity station join checks")
print("=" * 70)

print("beach_long shape:", beach_long.shape)
print("beach_stations shape:", beach_stations.shape)

print("\nStation ID checks")
print("=" * 70)

print("station_id in beach_long:", "station_id" in beach_long.columns)
print("station_id in beach_stations:", "station_id" in beach_stations.columns)

print("\nUnique station_id values")
print("=" * 70)
print("beach_long unique station_id:", beach_long["station_id"].nunique())
print("beach_stations unique station_id:", beach_stations["station_id"].nunique())

print("\nMissing station_id values")
print("=" * 70)
print("beach_long missing station_id:", beach_long["station_id"].isna().sum())
print("beach_stations missing station_id:", beach_stations["station_id"].isna().sum())

print("\nDuplicate station_id values in metadata")
print("=" * 70)
print("beach_stations duplicate station_id:", beach_stations["station_id"].duplicated().sum())

long_station_ids = set(beach_long["station_id"].dropna())
metadata_station_ids = set(beach_stations["station_id"].dropna())

missing_from_metadata = sorted(long_station_ids - metadata_station_ids)
extra_in_metadata = sorted(metadata_station_ids - long_station_ids)

print("\nStation ID coverage")
print("=" * 70)
print("Beach activity station IDs missing from metadata:", len(missing_from_metadata))
print("Metadata station IDs not used in beach activity long:", len(extra_in_metadata))

if len(missing_from_metadata) > 0:
    print("\nMissing from metadata preview:")
    print(missing_from_metadata[:20])

print("\nOverlapping column checks besides station_id")
print("=" * 70)

overlap_beach_station = sorted((set(beach_long.columns) & set(beach_stations.columns)) - {"station_id"})
print(overlap_beach_station)

print("\nbeach_long columns")
print("=" * 70)
print(list(beach_long.columns))

print("\nbeach_stations columns")
print("=" * 70)
print(list(beach_stations.columns))

Beach activity station join checks
beach_long shape: (12915, 36)
beach_stations shape: (284, 34)

Station ID checks
station_id in beach_long: True
station_id in beach_stations: True

Unique station_id values
beach_long unique station_id: 106
beach_stations unique station_id: 284

Missing station_id values
beach_long missing station_id: 0
beach_stations missing station_id: 0

Duplicate station_id values in metadata
beach_stations duplicate station_id: 0

Station ID coverage
Beach activity station IDs missing from metadata: 0
Metadata station IDs not used in beach activity long: 178

Overlapping column checks besides station_id
['station_lat', 'station_lon']

beach_long columns
['date', 'beach_event_id', 'start_date', 'end_date', 'start_date_clipped', 'end_date_clipped', 'duration_days_clean', 'analysis_period', 'post_emergency', 'study_group', 'impact_level', 'advisory_type_clean', 'advisory_cause_clean', 'source_clean', 'cause_group', 'beach_name_clean', 'station_id', 'station_name_cle

In [ ]:
beach_station_metadata_for_join = beach_stations.drop(
    columns=["station_lat", "station_lon"],
    errors="ignore"
)

beach_activity_station = beach_long.merge(
    beach_station_metadata_for_join,
    on="station_id",
    how="left"
)

beach_activity_station_path = DATA_PROCESSED / "beach_activity_station_analysis_ready.csv"

beach_activity_station.to_csv(beach_activity_station_path, index=False)

beach_activity_station_check = pd.read_csv(beach_activity_station_path)
beach_activity_station_check["date"] = pd.to_datetime(
    beach_activity_station_check["date"],
    errors="coerce"
)

print("Saved beach activity station analysis-ready file")
print("=" * 70)
print("Path:", beach_activity_station_path)
print("File exists:", beach_activity_station_path.exists())

print("\nReloaded QA")
print("=" * 70)
print("Rows:", len(beach_activity_station_check))
print("Columns:", beach_activity_station_check.shape[1])
print("Date range:", beach_activity_station_check["date"].min(), "to", beach_activity_station_check["date"].max())
print("Unique dates:", beach_activity_station_check["date"].nunique())

print("\nStation ID QA")
print("=" * 70)
print("Unique station_id values:", beach_activity_station_check["station_id"].nunique())
print("Missing station_id values:", beach_activity_station_check["station_id"].isna().sum())

metadata_key_fields = [
    "station_id",
    "station_name_clean",
    "station_lat",
    "station_lon",
    "beach_name",
    "nearest_city",
    "county_name",
    "agency_name",
    "coordinate_source",
    "valid_station_point",
]

existing_metadata_key_fields = [
    col for col in metadata_key_fields
    if col in beach_activity_station_check.columns
]

missing_metadata_key_fields = [
    col for col in metadata_key_fields
    if col not in beach_activity_station_check.columns
]

print("\nMissing expected key fields:")
print(missing_metadata_key_fields)

print("\nMissing values in existing key fields:")
print(beach_activity_station_check[existing_metadata_key_fields].isna().sum())

print("\nDuplicate column suffix check")
print("=" * 70)

suffix_cols = [
    col for col in beach_activity_station_check.columns
    if col.endswith("_x") or col.endswith("_y")
]

print("Columns ending in _x or _y:")
print(suffix_cols)

print("\nRows with missing metadata after join:")
metadata_check_cols = [
    col for col in ["beach_name", "nearest_city", "county_name", "agency_name"]
    if col in beach_activity_station_check.columns
]

if len(metadata_check_cols) > 0:
    missing_metadata_rows = beach_activity_station_check[metadata_check_cols].isna().all(axis=1).sum()
    print(missing_metadata_rows)
else:
    print("No metadata columns available for this check.")

print("\nFirst 5 rows:")
display(beach_activity_station_check.head())

Saved beach activity station analysis-ready file
Path: /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/beach_activity_station_analysis_ready.csv
File exists: True

Reloaded QA
Rows: 12915
Columns: 67
Date range: 2023-01-01 00:00:00 to 2026-03-04 00:00:00
Unique dates: 1159

Station ID QA
Unique station_id values: 106
Missing station_id values: 0

Missing expected key fields:
[]

Missing values in existing key fields:
station_id             0
station_name_clean     0
station_lat            0
station_lon            0
beach_name             0
nearest_city           0
county_name            0
agency_name            0
coordinate_source      0
valid_station_point    0
dtype: int64

Duplicate column suffix check
Columns ending in _x or _y:
[]

Rows with missing metadata after join:
0

First 5 rows:


,date,beach_event_id,start_date,end_date,start_date_clipped,end_date_clipped,duration_days_clean,analysis_period,post_emergency,study_group,...,regional_board,regional_board_name,coordinate_source,beach_upper_lat,beach_upper_lon,beach_lower_lat,beach_lower_lon,beach_mid_lat,beach_mid_lon,valid_station_point
0,2023-01-01,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,RB9,San Diego,station_upper,32.8079,-117.2662,32.8008,-117.2594,32.80435,-117.2628,True
1,2023-01-02,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,RB9,San Diego,station_upper,32.8079,-117.2662,32.8008,-117.2594,32.80435,-117.2628,True
2,2023-01-03,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,RB9,San Diego,station_upper,32.8079,-117.2662,32.8008,-117.2594,32.80435,-117.2628,True
3,2023-01-04,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,RB9,San Diego,station_upper,32.8079,-117.2662,32.8008,-117.2594,32.80435,-117.2628,True
4,2023-01-05,1,2023-01-01,2023-01-26,2023-01-01,2023-01-26,25,Pre-emergency,0,Comparison Beaches,...,RB9,San Diego,station_upper,32.8079,-117.2662,32.8008,-117.2594,32.80435,-117.2628,True


In [ ]:
output_paths = {
    "daily_environmental": DATA_PROCESSED / "daily_environmental_analysis_ready.csv",
    "h2s_daily_environmental": DATA_PROCESSED / "h2s_daily_environmental_analysis_ready.csv",
    "h2s_rtma_environmental": DATA_PROCESSED / "h2s_rtma_environmental_analysis_ready.csv",
    "beach_activity_station": DATA_PROCESSED / "beach_activity_station_analysis_ready.csv",
    "census_attributes": DATA_PROCESSED / "south_bay_census_tracts_proximity_attributes.csv",
}

qa_key_fields = {
    "daily_environmental": [
        "date", "rain_mm", "rain_lag1_mm", "rain_3day_mm", "rain_7day_mm",
        "rainy_day", "discharge_m3s", "discharge_cfs", "discharge_mgd",
        "high_flow_day", "flow_category", "flow_lag1_m3s",
        "flow_3day_mean_m3s", "flow_7day_mean_m3s",
        "high_flow_1day_window", "high_flow_3day_window", "high_flow_7day_window"
    ],
    "h2s_daily_environmental": [
        "date", "location_clean", "mean_h2s_ppb", "max_h2s_ppb",
        "elevated_h2s_events", "orange_h2s_events",
        "any_elevated_h2s_day", "any_orange_h2s_day",
        "rain_mm", "discharge_m3s", "high_flow_day", "flow_category"
    ],
    "h2s_rtma_environmental": [
        "date", "datetime_local", "datetime_utc", "location_clean", "h2s_ppb",
        "h2s_ge_10_event", "orange_h2s_event", "temperature_2m_f",
        "wind_speed_10m_mph", "wind_direction_10m", "rain_mm",
        "discharge_m3s", "high_flow_day", "flow_category"
    ],
    "beach_activity_station": [
        "date", "station_id", "station_name_clean", "station_lat", "station_lon",
        "beach_name", "nearest_city", "county_name", "agency_name",
        "coordinate_source", "valid_station_point"
    ],
    "census_attributes": [
        "geoid", "tract_name", "nearest_beach_station_km",
        "nearest_impact_station_km", "nearest_h2s_station_km",
        "tract_point_lat", "tract_point_lon"
    ],
}

print("Final QA for Notebook 09 outputs")
print("=" * 80)

loaded_outputs = {}

for name, path in output_paths.items():
    print(f"\n{name}")
    print("-" * 80)
    print("Path:", path)
    print("File exists:", path.exists())

    if name == "census_attributes":
        df = pd.read_csv(path, dtype={"geoid": str})
    else:
        df = pd.read_csv(path)

    loaded_outputs[name] = df

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    print("Rows:", len(df))
    print("Columns:", df.shape[1])

    if "date" in df.columns:
        print("Date range:", df["date"].min(), "to", df["date"].max())
        print("Unique dates:", df["date"].nunique())
        print("Missing dates:", df["date"].isna().sum())

    if name == "daily_environmental":
        print("Duplicate dates:", df["date"].duplicated().sum())

    if "station_id" in df.columns:
        print("Unique station_id values:", df["station_id"].nunique())
        print("Missing station_id values:", df["station_id"].isna().sum())

    if "location_clean" in df.columns:
        print("Unique location_clean values:", df["location_clean"].nunique())
        print("Missing location_clean values:", df["location_clean"].isna().sum())

    if "geoid" in df.columns:
        print("GEOID length counts:")
        print(df["geoid"].dropna().astype(str).str.len().value_counts().sort_index())

    expected_fields = qa_key_fields[name]
    existing_fields = [col for col in expected_fields if col in df.columns]
    missing_fields = [col for col in expected_fields if col not in df.columns]

    print("\nMissing expected key fields:")
    print(missing_fields)

    print("\nMissing values in existing key fields:")
    print(df[existing_fields].isna().sum())

    suffix_cols = [col for col in df.columns if col.endswith("_x") or col.endswith("_y")]
    print("\nColumns ending in _x or _y:")
    print(suffix_cols)

print("\nOutput files created")
print("=" * 80)

for name, path in output_paths.items():
    if name != "census_attributes":
        print(path.name)

Final QA for Notebook 09 outputs

daily_environmental
--------------------------------------------------------------------------------
Path: /content/gdrive_clean_mount/MyDrive/STAT596/Project596_datafiles/processed_data/daily_environmental_analysis_ready.csv
File exists: True
Rows: 1159
Columns: 77
Date range: 2023-01-01 00:00:00 to 2026-03-04 00:00:00
Unique dates: 1159
Missing dates: 0
Duplicate dates: 0

Missing expected key fields:
[]

Missing values in existing key fields:
date                     0
rain_mm                  0
rain_lag1_mm             1
rain_3day_mm             0
rain_7day_mm             0
rainy_day                0
discharge_m3s            0
discharge_cfs            0
discharge_mgd            0
high_flow_day            0
flow_category            0
flow_lag1_m3s            1
flow_3day_mean_m3s       0
flow_7day_mean_m3s       0
high_flow_1day_window    0
high_flow_3day_window    0
high_flow_7day_window    0
dtype: int64

Columns ending in _x or _y:
[]

h2s_daily_e